# 🏥 Diyabet KDS - MetaboNet Preprocessing (POLARS)
**Ultra Memory-Efficient Veri Hazırlama**

Bu notebook Polars kullanır:
- ⚡ Pandas'tan 10-100x hızlı
- 💾 2-5x daha az RAM
- 🦀 Rust tabanlı, lazy evaluation

88M+ satır için 12.7GB RAM yeterli!

## 1. Kurulum

In [ ]:
!pip install polars pyarrow -q

import polars as pl
import numpy as np
import pickle
import os
from pathlib import Path

print(f'Polars version: {pl.__version__}')

# RAM kontrolü
import psutil
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f'Available RAM: {ram_gb:.1f} GB')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/DiabetesKDS'
DATA_DIR = f'{WORK_DIR}/data'
OUTPUT_DIR = f'{DATA_DIR}/processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_FILE = f'{DATA_DIR}/metabonet_public_train.parquet'

if os.path.exists(TRAIN_FILE):
    size_gb = os.path.getsize(TRAIN_FILE) / (1024**3)
    print(f'✅ Dosya bulundu: {size_gb:.2f} GB')
else:
    print(f'❌ Dosya bulunamadı!')

## 2. Config

In [ ]:
CONFIG = {
    'cgm_col': 'CGM',
    'bolus_col': 'bolus',
    'carb_col': 'carbs',
    'time_col': 'date',
    'patient_col': 'id',
    'source_col': 'source_file',
    
    # Minimum gereksinimler
    'min_cgm_count': 1000,
    'min_bolus_events': 50,
    'min_carb_events': 50,
    
    # Output
    'feature_cols': ['CGM', 'carbs', 'cgm_delta_5min', 'time_sin', 'time_cos'],
}

print('Config yüklendi ✅')

## 3. Lazy Scan ile Parquet Oku

Polars `scan_parquet` RAM'e yüklemeden dosyayı okur (lazy evaluation).

In [ ]:
%%time

# Lazy scan - RAM kullanmadan okur
print('Parquet dosyası lazy scan ediliyor...')

lf = pl.scan_parquet(TRAIN_FILE)

# Schema göster
print(f'\nŞema:')
for col, dtype in lf.collect_schema().items():
    print(f'  {col}: {dtype}')

In [ ]:
%%time

# Unique source'ları bul (lazy ile)
print('Veri setleri analiz ediliyor...')

source_stats = (
    lf
    .group_by(CONFIG['source_col'])
    .agg([
        pl.len().alias('rows'),
        pl.col(CONFIG['cgm_col']).count().alias('cgm_count'),
        (pl.col(CONFIG['bolus_col']) > 0).sum().alias('bolus_events'),
        (pl.col(CONFIG['carb_col']) > 0).sum().alias('carb_events'),
    ])
    .collect()  # Burada execute edilir
)

print(source_stats)

In [ ]:
# Filtreleme
print('='*60)
print('FİLTRELEME RAPORU')
print('='*60)

valid_sources = []

for row in source_stats.iter_rows(named=True):
    source = row[CONFIG['source_col']]
    is_valid = True
    reasons = []
    
    if row['cgm_count'] < CONFIG['min_cgm_count']:
        is_valid = False
        reasons.append('CGM yetersiz')
    if row['bolus_events'] < CONFIG['min_bolus_events']:
        is_valid = False
        reasons.append('Bolus yetersiz')
    if row['carb_events'] < CONFIG['min_carb_events']:
        is_valid = False
        reasons.append('Carbs yetersiz')
    
    status = '✅' if is_valid else '❌'
    print(f"{status} {source}: {row['rows']:,} rows, {row['cgm_count']:,} CGM, "
          f"{row['bolus_events']:,} bolus, {row['carb_events']:,} carbs")
    
    if is_valid:
        valid_sources.append(source)

print(f'\n📊 {len(valid_sources)} veri seti kullanılacak')

## 4. Veri Çıkarma (Lazy + Streaming)

In [ ]:
%%time

print('Geçerli veriler çıkarılıyor (lazy)...')

# Lazy filtering
cols_to_select = [
    CONFIG['patient_col'],
    CONFIG['time_col'],
    CONFIG['source_col'],
    CONFIG['cgm_col'],
    CONFIG['bolus_col'],
    CONFIG['carb_col'],
]

lf_filtered = (
    lf
    .select(cols_to_select)
    .filter(pl.col(CONFIG['source_col']).is_in(valid_sources))
    .filter(pl.col(CONFIG['cgm_col']).is_not_null())
    .filter(pl.col(CONFIG['time_col']).is_not_null())
    .with_columns([
        pl.col(CONFIG['bolus_col']).fill_null(0),
        pl.col(CONFIG['carb_col']).fill_null(0),
    ])
)

print('Lazy plan oluşturuldu ✅')
print('\nPlan:')
print(lf_filtered.explain()[:500])

In [ ]:
%%time

# Streaming modunda collect (düşük RAM kullanımı)
print('Streaming modunda veri yükleniyor...')
print('⚠️ Bu işlem 5-15 dakika sürebilir...')

df = lf_filtered.collect(streaming=True)

print(f'\n✅ Yüklendi: {len(df):,} satır')
print(f'   RAM kullanımı: {df.estimated_size("gb"):.2f} GB')

In [ ]:
%%time

# Sıralama (Polars çok hızlı)
print('Sıralanıyor...')

df = df.sort([CONFIG['patient_col'], CONFIG['time_col']])

print(f'✅ Sıralama tamamlandı')
print(f'   Satır: {len(df):,}')
print(f'   Hasta: {df[CONFIG["patient_col"]].n_unique():,}')

## 5. Feature Engineering (Polars ile)

In [ ]:
%%time

print('Feature Engineering...')

df = df.with_columns([
    # Time of day (cyclical)
    (pl.col(CONFIG['time_col']).dt.hour() * 60 + pl.col(CONFIG['time_col']).dt.minute()).alias('minutes_of_day'),
])

df = df.with_columns([
    (2 * np.pi * pl.col('minutes_of_day') / (24 * 60)).sin().alias('time_sin'),
    (2 * np.pi * pl.col('minutes_of_day') / (24 * 60)).cos().alias('time_cos'),
    pl.col(CONFIG['time_col']).dt.weekday().alias('day_of_week'),
])

df = df.with_columns([
    (pl.col('day_of_week') >= 5).cast(pl.Int8).alias('is_weekend'),
])

print('  ✅ time_sin, time_cos, day_of_week, is_weekend')

# CGM deltas (per patient)
df = df.with_columns([
    pl.col(CONFIG['cgm_col']).diff().over(CONFIG['patient_col']).fill_null(0).alias('cgm_delta_5min'),
    pl.col(CONFIG['cgm_col']).diff(3).over(CONFIG['patient_col']).fill_null(0).alias('cgm_delta_15min'),
])

print('  ✅ cgm_delta_5min, cgm_delta_15min')

# Gereksiz sütunu kaldır
df = df.drop('minutes_of_day')

print(f'\n✅ Feature Engineering tamamlandı')
print(f'   Sütunlar: {df.columns}')
print(f'   RAM: {df.estimated_size("gb"):.2f} GB')

In [ ]:
# Veri örneği
print('Veri örneği:')
df.head(10)

## 6. Scaler Hesapla ve Kaydet

In [ ]:
# Scaler bilgileri (manuel hesapla, sklearn yerine)
print('Scaler istatistikleri hesaplanıyor...')

scalers = {}

for col in [CONFIG['cgm_col'], CONFIG['bolus_col'], CONFIG['carb_col']]:
    stats = df.select([
        pl.col(col).mean().alias('mean'),
        pl.col(col).std().alias('std'),
    ]).to_dicts()[0]
    
    scalers[col] = {
        'mean': stats['mean'],
        'std': stats['std'] if stats['std'] > 0 else 1.0
    }
    print(f"  {col}: mean={stats['mean']:.4f}, std={stats['std']:.4f}")

print('✅ Scalers hesaplandı')

## 7. Kaydet

In [ ]:
%%time

print('Veriler kaydediliyor...')

# Parquet olarak kaydet (Polars çok hızlı)
output_file = f'{OUTPUT_DIR}/metabonet_processed.parquet'
df.write_parquet(output_file, compression='zstd')
print(f'✅ {output_file}')

# Scalers
scalers_file = f'{OUTPUT_DIR}/scalers.pkl'
with open(scalers_file, 'wb') as f:
    pickle.dump(scalers, f)
print(f'✅ {scalers_file}')

# Config
config_file = f'{OUTPUT_DIR}/config.pkl'
with open(config_file, 'wb') as f:
    pickle.dump(CONFIG, f)
print(f'✅ {config_file}')

# Dosya boyutu
size_mb = os.path.getsize(output_file) / (1024**2)
print(f'\nÇıktı boyutu: {size_mb:.1f} MB')

In [ ]:
print('\n' + '='*60)
print('🎉 PREPROCESSING TAMAMLANDI!')
print('='*60)
print(f'\n📊 ÖZET:')
print(f'   Toplam satır: {len(df):,}')
print(f'   Benzersiz hasta: {df[CONFIG["patient_col"]].n_unique():,}')
print(f'   Veri setleri: {valid_sources}')
print(f'   RAM kullanımı: {df.estimated_size("gb"):.2f} GB')
print(f'\n   Kayıt konumu: {OUTPUT_DIR}')
print('\n👉 Şimdi 02_training_colab.ipynb notebook\'unu çalıştırabilirsin!')